# 20 — Fetch de vídeos
Atualiza a entidade de vídeo e grava seu snapshot de métricas.

In [ ]:
import json

from youtube_etl_genai.main import _get_api_key, _get_spark_session
from youtube_etl_genai.observability import TaskExecution, configure_job_logging
from youtube_etl_genai.pipeline import fetch_videos_step

TASK_KEY = "fetch_videos"
configure_job_logging()

for name, default in [("ingestion_id", ""), ("catalog", "youtube_lakehouse"), ("task_run_id", ""), ("secret_scope", "youtube_api_key"), ("secret_key", "api-key")]:
    dbutils.widgets.text(name, default)

spark = _get_spark_session()
catalog = dbutils.widgets.get("catalog")
ingestion_id = dbutils.widgets.get("ingestion_id")
with TaskExecution(spark=spark, catalog=catalog, task_key=TASK_KEY, task_run_id=dbutils.widgets.get("task_run_id") or None, ingestion_id=ingestion_id) as task_execution:
    result = fetch_videos_step(spark=spark, api_key=_get_api_key(spark, dbutils.widgets.get("secret_scope"), dbutils.widgets.get("secret_key")), ingestion_id=ingestion_id, catalog=catalog, api_cost_observer=task_execution.add_api_cost)
    task_execution.complete_from_result(result)
print(json.dumps(result, sort_keys=True))
